In [55]:
from prefect_s3_utils import connect_s3, upload_s3, download_s3
from prefect import flow, task
from prefect.blocks.system import Secret
import asyncio
import ast
import pandas as pd


@task(name="Download prior tagged Bills")
def get_tagged_bills() -> pd.DataFrame:
    """
    Downloads previously tagged bills from S3.

    Purpose: Avoids re-processing tags for bills that have already been handled.

    Returns:
        pd.DataFrame: A DataFrame containing previously tagged bills,
                      including a 'dataset_hash' column used for deduplication.
    """
    s3 = connect_s3('portfolio-project-files')

    tagged_bills = download_s3(s3, 'sdg-bill-tracking/tagged_bills.csv')

    corpus = download_s3(s3, 'sdg-bill-tracking/sdg_indicators_corpus.csv')
    corpus = corpus[['SDG No.', 'Target No.', 'SDG', 'Target']].drop_duplicates().reset_index(drop=True)
    
    return tagged_bills, corpus


@task(name="Upload new tagged bills to S3")
def upload_tagged_bills(tagged_bills_updated: pd.DataFrame) -> None:
    """
    Uploads the updated tagged bills DataFrame to S3.

    Args:
        tagged_bills_updated (pd.DataFrame): The updated DataFrame containing
                                             both old and newly tagged bills.
    """
    s3 = connect_s3('portfolio-project-files')
    upload_s3(s3, 'sdg-bill-tracking/tagged_bills.csv', tagged_bills_updated)


@flow(name='Inspection', log_prints=True)
async def bill_processing_main() -> None:


    tagged_bills, corpus = get_tagged_bills()


    return tagged_bills, corpus



tagged_bills, corpus = await bill_processing_main()

08:56:26.491 | INFO    | prefect.engine - View at https://app.prefect.cloud/account/3f9bd1dc-a34b-4ba7-a6e0-e2aa163f25d6/workspace/4fe3e874-7162-4c73-879d-d1f78dbe5925/runs/flow-run/06a54e07-a648-776b-8000-c81f2a4c2aaa

08:56:26.719 | INFO    | Flow run 'natural-skunk' - Beginning flow run 'natural-skunk' for flow 'Inspection'

08:56:26.722 | INFO    | Flow run 'natural-skunk' - View at https://app.prefect.cloud/account/3f9bd1dc-a34b-4ba7-a6e0-e2aa163f25d6/workspace/4fe3e874-7162-4c73-879d-d1f78dbe5925/runs/flow-run/06a54e07-a648-776b-8000-c81f2a4c2aaa

08:56:28.692 | INFO    | Task run 'Download prior tagged Bills-c6a' - Finished in state Completed()

08:56:28.883 | INFO    | Flow run 'natural-skunk' - Finished in state Completed()

In [56]:
# 14690

In [57]:
tagged_bills

,bill_file,dataset_hash,legislative_body,session_id,session_name,bill_number,url,status,status_date,title,description,sponsors,tagged_sdg,tagged_target,tagged_target_confidence
0,US/2025-2026_119th_Congress/bill/HB4.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,HB4,https://legiscan.com/US/bill/HB4/2025,Passed,2025-07-24,Rescissions Act of 2025,To rescind certain budget authority proposed t...,"['Steve Scalise (R)', 'Tom Cole (R)', 'Aaron B...",NaN,NaN,NaN
1,US/2025-2026_119th_Congress/bill/HB14.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,HB14,https://legiscan.com/US/bill/HB14/2025,Introduced,2025-03-05,John R. Lewis Voting Rights Advancement Act of...,To amend the Voting Rights Act of 1965 to revi...,"['Terri Sewell (D)', 'Hakeem Jeffries (D)', 'K...",NaN,NaN,NaN
2,US/2025-2026_119th_Congress/bill/HB22.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,HB22,https://legiscan.com/US/bill/HB22/2025,Engrossed,2025-04-10,SAVE Act Safeguard American Voter Eligibility Act,To amend the National Voter Registration Act o...,"['Chip Roy (R)', 'Andrew Garbarino (R)', 'Nico...",NaN,NaN,NaN
3,US/2025-2026_119th_Congress/bill/HB29.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,HB29,https://legiscan.com/US/bill/HB29/2025,Engrossed,2025-01-08,Laken Riley Act,To require the Secretary of Homeland Security ...,"['Mike Collins (R)', 'Rick Allen (R)', 'Marjor...",NaN,NaN,NaN
4,US/2025-2026_119th_Congress/bill/HB32.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,HB32,https://legiscan.com/US/bill/HB32/2025,Introduced,2025-01-03,No Bailout for Sanctuary Cities Act,To provide that sanctuary jurisdictions that p...,"['Nick LaLota (R)', 'Randy Feenstra (R)', 'Eri...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17319,US/2025-2026_119th_Congress/bill/SJR144.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,SJR144,https://legiscan.com/US/bill/SJR144/2025,Introduced,2026-03-25,A joint resolution providing for congressional...,A joint resolution providing for congressional...,['Sheldon Whitehouse (D)'],5.0,5.4,NaN
17320,US/2025-2026_119th_Congress/bill/SJR152.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,SJR152,https://legiscan.com/US/bill/SJR152/2025,Introduced,2026-03-26,A joint resolution providing for congressional...,A joint resolution providing for congressional...,['Alex Padilla (D)'],8.0,5.4,NaN
17321,US/2025-2026_119th_Congress/bill/SJR155.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,SJR155,https://legiscan.com/US/bill/SJR155/2025,Introduced,2026-03-26,A joint resolution providing for congressional...,A joint resolution providing for congressional...,['Sheldon Whitehouse (D)'],5.0,5.4,NaN
17322,US/2025-2026_119th_Congress/bill/SJR156.json,dc1e23fde1e2b566c4f7a68f4164cf5d,US,2199,119th Congress,SJR156,https://legiscan.com/US/bill/SJR156/2025,Introduced,2026-03-26,A joint resolution providing for congressional...,A joint resolution providing for congressional...,['Jeff Merkley (D)'],5.0,5.4,NaN
